# Random Forest Klassifikation für KHK / Herzinfarkt (CHD/MI)
### Foundations of Data Science — Gruppenprojekt (NHANES 2021–2023)
**Autor:** Marco (Assignment: Random Forest)

---

## 1. Forschungsfrage und Zielsetzung
Verbessern die systemischen Entzündungsmarker **NLR (Neutrophil-to-Lymphocyte Ratio)** und **hs-CRP (High-Sensitivity C-Reactive Protein)** die Vorhersage und Diskrimination von koronarer Herzkrankheit bzw. Myokardinfarkt (CHD/MI) **über etablierte kardiovaskuläre Risikofaktoren hinaus**?

### Warum Random Forest? (Vorlesung Woche 7)
Ein **Random Forest** ist ein Ensemble-Modell aus vielen einzelnen **Entscheidungsbäumen (Decision Trees)**:
1. **Nichtlinearität & Interaktionen:** Er kann komplexe, nichtlineare Zusammenhänge und Wechselwirkungen zwischen Risikofaktoren erfassen, ohne dass wir diese manuell vorgeben müssen.
2. **Bagging (Bootstrap Aggregation):** Jeder Baum wird auf einer zufälligen Stichprobe mit Zurücklegen trainiert. Das senkt die Varianz drastisch gegenüber einem einzelnen überangepassten Baum.
3. **Zufällige Feature-Auswahl (Random Subspace):** An jedem Split wird nur eine zufällige Teilmenge an Variablen betrachtet (`max_features='sqrt'`). Dadurch werden die Bäume dekorreliert.
4. **Mehrheitsentscheid / Mittelung:** Alle 300 Bäume stimmen ab, und der Durchschnitt ihrer Voten ergibt die vorhergesagte Wahrscheinlichkeit.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Pfad zum Projekt-Root
ROOT = Path.cwd().parents[1] if Path.cwd().name == "marco" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from prepare_data import CONFIG, load_data

OUT = ROOT / "notebooks" / "marco" / "results"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

SEED = CONFIG["random_state"]
print(f"Setup abgeschlossen! Random Seed: {SEED}")

## 2. Daten laden und Feature-Sets definieren
Wir nutzen denselben festen 80/20 Train/Test-Split wie die gesamte Gruppe:
- **Train-Set:** $n=3{,}635$ Probanden (270 Fälle, 7.4%)
- **Test-Set:** $n=909$ Probanden (67 Fälle, 7.4%)

Die hierarchisch geschachtelten Modelle:
- **M0:** Basis-Demographie (`age`, `male`)
- **M1:** Soziodemographie (`M0` + `bmi`, `race_eth`, `income_pir`)
- **M2:** Etablierte Risikofaktoren (`M1` + `smoking`, `diabetes`, `hypertension`, `total_chol`, `hdl_chol`, `egfr`)
- **M2 + NLR:** `M2` + log-transformiertes NLR
- **M2 + CRP:** `M2` + log-transformiertes hs-CRP
- **M3 (Vollmodell):** `M2` + beide Entzündungsmarker (`log_nlr`, `log_crp`)
- **Sensitivitätsmodelle ohne Cholesterin:** `M2 no chol` und `M3 no chol`

In [ ]:
M0 = ["age", "male"]
M1 = M0 + ["bmi", "race_eth", "income_pir"]
M2 = M1 + ["smoking", "diabetes", "hypertension", "total_chol", "hdl_chol", "egfr"]
CHOLESTEROL = ["total_chol", "hdl_chol"]
M2_NO_CHOL = [f for f in M2 if f not in CHOLESTEROL]

FEATURE_SETS = {
    "M0": M0,
    "M1": M1,
    "M2": M2,
    "M2 + NLR": M2 + ["log_nlr"],
    "M2 + CRP": M2 + ["log_crp"],
    "M3": M2 + ["log_nlr", "log_crp"],
    "M2 no chol": M2_NO_CHOL,
    "M3 no chol": M2_NO_CHOL + ["log_nlr", "log_crp"],
}
CATEGORICAL = ["race_eth", "smoking"]
BINARY = ["male", "diabetes", "hypertension"]
OUTCOME = "chd_mi"

train, test = load_data(split=True)
y_train, y_test = train[OUTCOME].astype(int), test[OUTCOME].astype(int)
print(f"Train n={len(train)} (Fälle: {y_train.sum()} = {y_train.mean():.1%})")
print(f"Test  n={len(test)} (Fälle: {y_test.sum()} = {y_test.mean():.1%})")

## 3. Geladene Hyperparameter aus der 5-Fold Cross-Validation
Die besten Parameter wurden mit Stratified 5-Fold Cross-Validation auf dem Trainings-Set ermittelt:

In [ ]:
cv_results = pd.read_csv(OUT / "cv_results.csv")
display(cv_results)

## 4. Test-Performance & Added Value der Marker ($\Delta$AUC)
Ergebnisse auf dem unabhängigen Test-Set ($n=909$) mit 1.000 Bootstrap-Resamples für 95%-Konfidenzintervalle:

In [ ]:
test_perf = pd.read_csv(OUT / "test_performance.csv")
print("=== Test-Performance aller Feature-Sets ===")
display(test_perf.round(4))

delta_auc = pd.read_csv(OUT / "delta_auc.csv")
print("\n=== Mehrwert der Marker (Delta-AUC mit 95% Bootstrap CI) ===")
display(delta_auc.round(4))

## 5. Permutation Feature Importance
Welche Variablen beeinflussen die Entscheidungen des Random Forest am stärksten?

In [ ]:
imp = pd.read_csv(OUT / "permutation_importance_m3.csv").sort_values("auc_drop_mean", ascending=False)
display(imp.round(4))

## 6. Generierte Publikations-Abbildungen anzeigen

In [ ]:
from IPython.display import Image, display

print("1. Diskrimination nach Feature-Set:")
display(Image(filename=str(FIG / "auc_by_feature_set.png")))

print("2. ROC- und Precision-Recall-Kurven (M2 vs. M3):")
display(Image(filename=str(FIG / "roc_pr_m2_m3.png")))

print("3. Permutation Feature Importance:")
display(Image(filename=str(FIG / "permutation_importance_m3.png")))

print("4. Partial Dependence der Entzündungsmarker:")
display(Image(filename=str(FIG / "partial_dependence_markers_m3.png")))